In [1]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


In [2]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

{"object":"list","data":[{"id":"Qwen3-30B-A3B","object":"model","created":1781450262,"owned_by":"vllm","root":"Qwen/Qwen3-30B-A3B","parent":null,"max_model_len":40960,"permission":[{"id":"modelperm-4cc6870e59764bf886328ea7494acffc","object":"model_permission","created":1781450262,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [3]:
!pip install -q pydantic-ai-slim openai


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [4]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

In [5]:
agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

In [6]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

In [8]:
pip install "pydantic-ai-slim[mcp]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 3.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 26.2 MB/s  0:00:00
  Attempting uninstall: pyjwt
    Found existing installation: PyJWT 2.3.0
    Uninstalling PyJWT-2.3.0:
      Successfully uninstalled PyJWT-2.3.0
  Attempting uninstall: starlette━━━━━━━━━━━━━━━━━━━━━━━━━  7/19 [beartype]
    Found existing installation: starlette 0.48.0━━━━━━━━━━━━━  7/19 [beartype]
    Uninstalling starlette-0.48.0:━━━━━━━━━━━━━━━━━━━━━━━  8/19 [starlette]
      Successfully uninstalled starlette-0.48.0━━━━━━━━━━━━━━━  8/19 [starlette]
  Attempting uninstall: keyring━━━━━━━━━━━━━━━━━━━━━━━  8/19 [starlette]
    Found existing installation: keyring 23.5.0m━━━━━━━━━━━━━━━━━━ 10/19 [keyring]
    Uninstalling keyring-23.5.0:0m╺━━━━━━━━━━━━━━━━━━ 10/19 [keyring]
      Successfully uninstalled keyring-23.5.0━━━━━━━━━━━━━━━━━ 10/19 [keyring]
  Attempti

In [9]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


In [10]:
await run_async("What is the capital of France?")

'\n\nThe capital of France is **Paris**. It is a major city in Europe and is renowned for its cultural landmarks, such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Paris serves as the political, economic, and cultural center of the country.'

In [11]:
await run_async("What’s the date today?")

'\n\nI don\'t have access to real-time data, so I can\'t provide the current date or time. However, you can check your device\'s clock or search online for "today\'s date" to get the most accurate information. Let me know if you need help with anything else! 😊'

In [12]:
from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [13]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

In [14]:
await run_async("What’s the date today?")

"\n\nToday's date is **Monday, June 14, 2026**."

In [15]:
!pip install -q mcp-server-time


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [16]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)

/tmp/ipykernel_4975/3040644900.py:3: DeprecationWarning: `MCPServerStdio` is deprecated and will be removed in v2. Use `MCPToolset('path/to/script.py')` for Python scripts, `MCPToolset('script.js')` for Node scripts, or `MCPToolset(fastmcp.client.transports.StdioTransport(command='...', args=[...]))` for arbitrary commands.
  time_server = MCPServerStdio(


In [17]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

In [18]:
await run_async("What’s the date today?")

'\n\nThe current date is **June 14, 2026**. \n\nThe current time in America/New_York is **11:23 AM** (Daylight Saving Time). Let me know if you need further details!'

In [19]:
# Install Node.js 20 via NodeSource
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt install -y nodejs

2026-06-14 15:24:06 - Installing pre-requisites
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:3 https://repo.radeon.com/amdgpu/7.0/ubuntu jammy InRelease [3183 B]       
Get:4 https://repo.radeon.com/rocm/apt/7.0 jammy InRelease [2603 B]            
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4006 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1303 kB]
Get:9 https://repo.radeon.com/amdgpu/7.0/ubuntu jammy/main amd64 Packages [1329 B]m
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]m
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7183 kB]
Get:12 http://archive.ubuntu.com/ubuntu

In [20]:
!node -v && npm -v && npx --version

v20.20.2
10.8.2
10.8.2


In [21]:
airbnb_server = MCPServerStdio(
    "npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"]
)

/tmp/ipykernel_4975/1537382601.py:1: DeprecationWarning: `MCPServerStdio` is deprecated and will be removed in v2. Use `MCPToolset('path/to/script.py')` for Python scripts, `MCPToolset('script.js')` for Node scripts, or `MCPToolset(fastmcp.client.transports.StdioTransport(command='...', args=[...]))` for arbitrary commands.
  airbnb_server = MCPServerStdio(


In [22]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server],
    system_prompt=system_prompt,
)


In [23]:
await run_async("Find a place to stay in Mumbai for next Sunday for 3 nights for 2 adults?")

'\n\nHere are some Airbnb listings in Mumbai for your stay from June 21 to June 24, 2026 (3 nights):\n\n1. **Amalfi 1 BHK in BKC**  \n   - **Price**: $229 CAD total  \n   - **Rating**: 4.64/5 (106 reviews)  \n   - **Details**: 1 bedroom, 1 queen bed, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms/1636113240871181895)\n\n2. **Cozy 1BHK In Bandra W**  \n   - **Price**: $241 CAD total  \n   - **Rating**: 4.94/5 (17 reviews)  \n   - **Details**: 1 bedroom, 1 double bed, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms/1594801114103667377)\n\n3. **Tranquil 2BHK Apt in BKC**  \n   - **Price**: $319 CAD total (originally $368 CAD)  \n   - **Rating**: 4.93/5 (147 reviews)  \n   - **Details**: 2 bedrooms, 2 beds, 2 baths  \n   - [View Listing](https://www.airbnb.com/rooms/1205161066654778300)\n\n4. **Victoria (Private Studio Apartment Bandra West)**  \n   - **Price**: $199 CAD total  \n   - **Rating**: 4.96/5 (69 reviews)  \n   - **Details**: 1 bedroom, 1 bath  \n   - [Vie